# Reconocimiento de gestos por posicion de puntos

In [ ]:
!pip install mediapipe opencv-python numpy

import cv2
import mediapipe as mp
import math
import numpy as np
import os
from google.colab.patches import cv2_imshow
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# La clase 'HandGestureDetector' tiene toda la lógica para la detección y
# clasificación de gestos de la mano a partir de una imagen.
class HandGestureDetector:
    def __init__(self):
        # Inicializa el modelo de detección de manos
        self.mp_hands = mp.solutions.hands
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        self.hands = self.mp_hands.Hands(
            static_image_mode=True,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    def calculate_finger_extensions(self, landmarks):
        # Determina el estado (extendido o flexionado) de cada dedo.
        # La lógica se basa en comparar la distancia desde la muñeca a la punta del dedo
        # con la distancia desde la muñeca a la articulación intermedia.
        # Si la punta está más alejada que la articulación, se considera extendido.

        def is_finger_extended(tip, pip):
            dist_tip_wrist = math.sqrt((tip.x - landmarks[0].x)**2 + (tip.y - landmarks[0].y)**2)
            dist_pip_wrist = math.sqrt((pip.x - landmarks[0].x)**2 + (pip.y - landmarks[0].y)**2)
            return dist_tip_wrist > dist_pip_wrist

        def is_thumb_extended(tip, ip):
            dist_tip_wrist = math.sqrt((tip.x - landmarks[0].x)**2 + (tip.y - landmarks[0].y)**2)
            dist_ip_wrist = math.sqrt((ip.x - landmarks[0].x)**2 + (ip.y - landmarks[0].y)**2)
            return dist_tip_wrist > dist_ip_wrist

        finger_extensions = {
            'thumb': is_thumb_extended(landmarks[4], landmarks[3]),
            'index': is_finger_extended(landmarks[8], landmarks[6]),
            'middle': is_finger_extended(landmarks[12], landmarks[10]),
            'ring': is_finger_extended(landmarks[16], landmarks[14]),
            'pinky': is_finger_extended(landmarks[20], landmarks[18])
        }
        return finger_extensions

    def is_ok_gesture(self, landmarks, finger_extensions):
        # Verifica específicamente el gesto "OK". Este gesto es positivo si
        # la distancia entre la punta del pulgar y el índice es muy pequeña y los
        # otros tres dedos (medio, anular y meñique) están extendidos.
        thumb_tip = landmarks[4]
        index_tip = landmarks[8]
        thumb_index_dist = math.sqrt((thumb_tip.x - index_tip.x)**2 + (thumb_tip.y - index_tip.y)**2)

        return (thumb_index_dist < 0.05 and
                finger_extensions['middle'] and
                finger_extensions['ring'] and
                finger_extensions['pinky'] and
                not finger_extensions['index'])

    def detect_gesture(self, landmarks, finger_extensions):
        # Clasifica el gesto de la mano basándose en reglas sobre los dedos extendidos.
        if self.is_ok_gesture(landmarks, finger_extensions):
            return "OK"

        is_index_ext = finger_extensions['index']
        is_middle_ext = finger_extensions['middle']
        is_ring_ext = finger_extensions['ring']
        is_pinky_ext = finger_extensions['pinky']

        if is_index_ext and is_middle_ext and not is_ring_ext and not is_pinky_ext:
            return "PAZ"
        elif is_index_ext and is_middle_ext and is_ring_ext and is_pinky_ext:
            return "PALMA"
        elif not is_index_ext and not is_middle_ext and not is_ring_ext and not is_pinky_ext:
            return "PUÑO"
        else:
            return "GESTO_DESCONOCIDO"

    def process_image(self, image):
        # Función principal, convierte la imagen a RGB, la pasa al modelo de MediaPipe, y si se
        # detectan manos, extrae los landmarks, determina el gesto y dibuja los resultados sobre
        # una copia de la imagen original.
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb_image)
        output_image = image.copy()

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # 1. Calcular el estado de los dedos
                finger_extensions = self.calculate_finger_extensions(hand_landmarks.landmark)

                # 2. Clasificar el gesto
                gesture = self.detect_gesture(hand_landmarks.landmark, finger_extensions)

                # 3. Dibujar los landmarks en la imagen de salida
                self.mp_drawing.draw_landmarks(
                    output_image,
                    hand_landmarks,
                    self.mp_hands.HAND_CONNECTIONS,
                    self.mp_drawing_styles.get_default_hand_landmarks_style(),
                    self.mp_drawing_styles.get_default_hand_connections_style()
                )

                # 4. Añadir el texto con el gesto detectado
                cv2.putText(output_image, gesture, (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)

        return output_image

# Este bloque de código se ejecuta al correr el script. Su función es
# instanciar el detector, definir la ruta del dataset de imágenes y
# procesar cada imagen una por una, mostrando el resultado final.
if __name__ == "__main__":
    detector = HandGestureDetector()
    dataset_path = "/content/drive/MyDrive/Colab Notebooks/TDP2/Dataset"

    if not os.path.exists(dataset_path):
        print(f"Error: La ruta especificada no existe -> {dataset_path}")
    else:
        image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for image_file in sorted(image_files):
            image_path = os.path.join(dataset_path, image_file)
            image = cv2.imread(image_path)

            if image is None:
                print(f"Advertencia: No se pudo leer la imagen {image_file}")
                continue

            # Procesar la imagen para detectar y clasificar el gesto
            processed_image = detector.process_image(image)